# 04b Per-choice scoring
Additional check, not required by the project: model 1 with another input format. The selected models stay the chosen runs of `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb`.

**In short.** Question: does scoring each hypothesis with the same classifier lower the validation loss of model 1? Answer: on the validation set, yes: 0.5858 and 68.90% against 0.6460 and 64.00%, with significantly more correct predictions (p = 0.0029). It is one seed, so this does not show that per-choice is better in general.

The settings of the chosen model 1, which were tuned for fixed positions: 600 hidden units, learning rate 0.000316, batch size 2048, patience 10, at most 300 epochs, seed 42. It is one run and a report, not a candidate. The test set is not loaded here; `05_evaluation.ipynb` evaluates this run once, as a separate comparison.

In [1]:
# setup: on Colab, connect Google Drive; everywhere, use the GPU if there is one
import os
import sys

# on a GPU, a seed alone does not repeat a run exactly; this variable must be set before the GPU is used
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:2"

import torch

torch.backends.cudnn.deterministic = True
# deterministic operations where they exist, and a warning for the others
torch.use_deterministic_algorithms(True, warn_only=True)

if "google.colab" in sys.modules:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/nlp-project1-art")
    %pip install -q wandb
    # the key comes from Colab Secrets, so it is not in the notebook
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
# the W&B server of this project
os.environ["WANDB_BASE_URL"] = "https://api.forge.coreweave.com"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Mounted at /content/drive
cuda Tesla T4


## Data
The features and labels saved in `02_preprocessing.ipynb`, as in `03_model1_embeddings.ipynb`: the input stays the four concatenated sentence vectors.

In [2]:
import numpy as np

data = np.load("features/model1.npz")
X_train = torch.tensor(data["X_train"]).to(device)
y_train = torch.tensor(data["y_train"]).to(device)
X_valid = torch.tensor(data["X_valid"]).to(device)
y_valid = torch.tensor(data["y_valid"]).to(device)
print(X_train.shape, X_valid.shape)

torch.Size([16801, 1200]) torch.Size([1000, 1200])


## Per-choice classifier
The chosen model 1 gives h1 and h2 fixed input positions, so its predictions can depend on the position (`04_model2_rnn.ipynb`, section "Check: does the position of the hypotheses matter?"). Here the same 2-layer classifier scores [O1, O2, h1] and [O1, O2, h2], with 1 output each: the pairs on slide 25 and the one score per choice on slide 26 of the course slides "Project Discussions" (Week 5), and for averaged GloVe vectors also the bag-of-words baseline of Bhagavatula et al. (2020, Appendix A.4), 50.5%. The two scores are the 2 logits, so the loss, the training and the evaluation do not change.

In [3]:
import torch.nn as nn

# same class as in 03_model1_embeddings.ipynb
class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

# one score per hypothesis: the same classifier scores each hypothesis
class PerChoiceClassifier(nn.Module):

    def __init__(self, text_dim, inner_dim):
        super().__init__()
        # size of the vector of one text (300)
        self.text_dim = text_dim
        # input: O1, O2 and one hypothesis; output: 1 score
        self.scorer = Classifier(3 * text_dim, 1, inner_dim=inner_dim)

    def forward(self, x):
        # x side by side: O1, O2, h1, h2
        observations = x[:, :2 * self.text_dim]
        h1 = x[:, 2 * self.text_dim:3 * self.text_dim]
        h2 = x[:, 3 * self.text_dim:]
        # the same scorer for both hypotheses: [batch size, 1] each
        score1 = self.scorer(torch.concat([observations, h1], dim=1))
        score2 = self.scorer(torch.concat([observations, h2], dim=1))
        # the two scores are the 2 logits: [batch size, 2], as in model 1
        return torch.concat([score1, score2], dim=1)

Check on 5 made-up examples: swapping h1 and h2 must swap the two logits.

In [4]:
# untrained classifier on 5 made-up examples: 4 texts of 300 numbers side by side
torch.manual_seed(42)
test_model = PerChoiceClassifier(300, inner_dim=600)
x = torch.randn(5, 1200)
# the same examples with h1 and h2 swapped
swapped = torch.concat([x[:, :600], x[:, 900:], x[:, 600:900]], dim=1)
with torch.no_grad():
    logits = test_model(x)
    swapped_logits = test_model(swapped)
# 2 logits per example, as in model 1
print(logits.shape)
# swapping the hypotheses must swap the logits
print(torch.equal(logits.flip(1), swapped_logits))
# number of parameters: per-choice (900 inputs, 1 output, used twice) and model 1 (1200 inputs, 2 outputs)
print(sum(p.numel() for p in test_model.parameters()), sum(p.numel() for p in Classifier(1200, 2, inner_dim=600).parameters()))

torch.Size([5, 2])
True
541201 721802


## Training
The same loss, batches, training and early stopping as in the sweep of `03_model1_embeddings.ipynb`, with the settings of the chosen model 1. `evaluate()` also returns the predictions, as in `05_evaluation.ipynb`, for the comparison below.

In [5]:
from torch.utils.data import DataLoader, TensorDataset

criterion = nn.CrossEntropyLoss()

# same function as in 03_model1_embeddings.ipynb; it also returns the predictions, as in 05_evaluation.ipynb
def evaluate(model, loader):
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            losses.append(criterion(output, y).item())
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy, predictions

# the settings of the chosen model 1
INNER_DIM = 600
LEARNING_RATE = 0.000316
BATCH_SIZE = 2048
MAX_EPOCHS = 300
PATIENCE = 10

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
# equal batches for the evaluation, as in 03_model1_embeddings.ipynb
train_eval_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=317)
valid_loader = DataLoader(TensorDataset(X_valid, y_valid), batch_size=100)

In [6]:
# no W&B log lines in the notebook output; the metrics are still logged
os.environ["WANDB_SILENT"] = "true"

import wandb
from torch.optim import Adam

wandb.login()

name = f"glove-mlp-per-choice-h{INNER_DIM}-lr{LEARNING_RATE}-b{BATCH_SIZE}"
checkpoint_path = f"checkpoints/{name}.pth"
wandb.init(
    project="nlp-project1-art-gpu",
    name=name,
    config={"model": "glove-mlp-per-choice", "inner_dim": INNER_DIM, "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "seed": 42},
)

torch.manual_seed(42)
model = PerChoiceClassifier(300, inner_dim=INNER_DIM).to(device)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
best_valid_loss = float("inf")
# number of epochs in a row without a new lowest validation loss
epochs_without_improvement = 0

for epoch in range(MAX_EPOCHS):
    model.train()
    for x, y in train_loader:
        output = model(x)
        loss = criterion(output, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss, train_acc, _ = evaluate(model, train_eval_loader)
    valid_loss, valid_acc, _ = evaluate(model, valid_loader)
    wandb.log({"epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc, "val_loss": valid_loss, "val_acc": valid_acc})
    if valid_loss < best_valid_loss:
        # new lowest validation loss: save this epoch
        best_valid_loss = valid_loss
        best_valid_acc = valid_acc
        best_epoch = epoch + 1
        torch.save(model.state_dict(), checkpoint_path)
        epochs_without_improvement = 0
    else:
        # early stopping: no new lowest validation loss for PATIENCE epochs in a row
        epochs_without_improvement += 1
        if epochs_without_improvement == PATIENCE:
            break

# best values in the run summary, so that the runs can be sorted by them in W&B
wandb.run.summary["best_val_loss"] = best_valid_loss
wandb.run.summary["best_val_acc"] = best_valid_acc
wandb.run.summary["best_epoch"] = best_epoch
wandb.finish()
print(f"{name}: best epoch {best_epoch} of {epoch + 1}, train acc at the end {train_acc:.2%}, valid loss {best_valid_loss:.4f}, valid acc {best_valid_acc:.2%}")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


glove-mlp-per-choice-h600-lr0.000316-b2048: best epoch 97 of 107, train acc at the end 87.01%, valid loss 0.5858, valid acc 68.90%


## Comparison with the chosen model 1
Both are scored on the same 1000 validation examples, so they are compared with the exact McNemar test (McNemar, 1947), as in `04_model2_rnn.ipynb`.

In [7]:
from scipy.stats import binomtest

# the chosen model 1 of 03_model1_embeddings.ipynb
chosen = Classifier(1200, 2, inner_dim=INNER_DIM).to(device)
chosen.load_state_dict(torch.load("checkpoints/glove-mlp-sweep-h600-lr0.000316-b2048.pth", map_location=device, weights_only=True))
# the best epoch of the run above
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
chosen_loss, chosen_acc, chosen_predictions = evaluate(chosen, valid_loader)
per_choice_loss, per_choice_acc, per_choice_predictions = evaluate(model, valid_loader)
print(f"chosen model 1: valid loss {chosen_loss:.4f}, valid acc {chosen_acc:.2%}")
print(f"per-choice: valid loss {per_choice_loss:.4f}, valid acc {per_choice_acc:.2%}")

# True where the prediction is right
correct_chosen = chosen_predictions == y_valid
correct_per_choice = per_choice_predictions == y_valid
# & = both True, ~ = not
only_per_choice = (correct_per_choice & ~correct_chosen).sum().item()
only_chosen = (~correct_per_choice & correct_chosen).sum().item()
# exact McNemar test: binomial test on the examples that only one of the two gets right
print(f"only per-choice right {only_per_choice}, only chosen right {only_chosen}, p = {binomtest(only_per_choice, only_per_choice + only_chosen, 0.5).pvalue:.4f}")

chosen model 1: valid loss 0.6460, valid acc 64.00%
per-choice: valid loss 0.5858, valid acc 68.90%
only per-choice right 155, only chosen right 106, p = 0.0029


## Result
- **Against the chosen model 1:** best epoch 97 of 107, validation loss 0.5858 and accuracy 68.90% against 0.6460 and 64.00%. Only per-choice right on 155 examples, only the chosen run on 106, exact McNemar test p = 0.0029.
- **Against the selected model 2** (0.5867, 66.70%): In this run, per-choice model 1 has slightly lower validation loss and higher accuracy than the selected model 2; this comparison was not statistically tested. So the gap between the models may depend on the input format.
- **Overfitting:** a gap remains at the best epoch: training accuracy 84.98%, validation accuracy 68.90% (W&B).
- **Limits:** one seed, with settings tuned for fixed positions. The validation set is unfiltered and the test set adversarially filtered (see "Data" in `README.md`), so the gain may not carry over. `05_evaluation.ipynb` reports it whatever the result.
- **Repeat:** all 107 epochs are identical to the run `prep-mlp-per-choice` of an earlier, longer version of this notebook (commit `9106efb`; W&B).

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Per-choice scoring: pairs per choice (slide 25) and one score per choice (slide 26) from the previous semester's examples in the course slides "Project Discussions" (Week 5); for averaged GloVe vectors also the bag-of-words baseline of Bhagavatula et al. (2020), Appendix A.4, https://arxiv.org/abs/1908.05739; `torch.concat` as in the course exercise "Sequence-to-sequence RNN" (Week 4); column indexing as in the course exercise "PyTorch" (Week 1).
- Check of the class: `torch.randn`, https://docs.pytorch.org/docs/stable/generated/torch.randn.html, `torch.flip`, https://docs.pytorch.org/docs/stable/generated/torch.flip.html, `torch.equal`, https://docs.pytorch.org/docs/stable/generated/torch.equal.html, and `torch.numel`, https://docs.pytorch.org/docs/stable/generated/torch.numel.html, from the PyTorch documentation.
- Features, `Classifier`, loss, optimizer, batches, seed, training loop, early stopping, checkpoint and W&B logging: as in `03_model1_embeddings.ipynb`, section "Sweep" (see the sources there, for `np.load`, `torch.tensor`, `nn.CrossEntropyLoss`, `torch.manual_seed`, `Adam`, `DataLoader`, `TensorDataset`, `float("inf")`, `torch.save`, `load_state_dict`, `wandb.login`, `wandb.init`, `wandb.log`, `wandb.run.summary` and `wandb.finish`).
- `evaluate()` with the predictions: as in `05_evaluation.ipynb` (see the sources there).
- Comparison with the chosen model 1: exact McNemar test (McNemar, 1947) with `scipy.stats.binomtest`, https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binomtest.html, as in `04_model2_rnn.ipynb` (see the sources there); `&` and `~` on True/False tensors from the PyTorch documentation `torch.bitwise_and` and `torch.bitwise_not`, https://docs.pytorch.org/docs/stable/generated/torch.bitwise_and.html and https://docs.pytorch.org/docs/stable/generated/torch.bitwise_not.html; `map_location` of `torch.load`, https://docs.pytorch.org/docs/stable/generated/torch.load.html.
- GPU: `torch.device`, `torch.cuda.is_available` and `Tensor.to`, PyTorch documentation "CUDA semantics", https://docs.pytorch.org/docs/stable/notes/cuda.html. Google Colab from the course slides "Course Project HS26", slide 11. Environment variables on Colab: Google Colab Secrets and `drive.mount`, https://colab.research.google.com/. Repeated runs on a GPU: `torch.backends.cudnn.deterministic`, https://docs.pytorch.org/docs/stable/backends.html, and PyTorch documentation "Reproducibility", https://docs.pytorch.org/docs/stable/notes/randomness.html; `torch.use_deterministic_algorithms` (`warn_only`), https://docs.pytorch.org/docs/stable/generated/torch.use_deterministic_algorithms.html; `CUBLAS_WORKSPACE_CONFIG`, from the warning about RNN functions in `torch.nn.LSTM`, https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html. Setup on Colab: `sys.modules`, https://docs.python.org/3/library/sys.html#sys.modules, and `os.chdir`, https://docs.python.org/3/library/os.html#os.chdir, Python documentation; `%pip`, IPython documentation, https://ipython.readthedocs.io/en/stable/interactive/magics.html.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md.
- Output of W&B: `WANDB_SILENT` (no W&B log lines in the notebook output), W&B documentation "Environment variables", https://docs.coreweave.com/models/track/environment-variables; `os.environ`, Python documentation, https://docs.python.org/3/library/os.html#os.environ.